# Writing a class-based signature


## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Adding instructional nuance with class-based signatures

A class-based signature details the same structure a string signature can. We have the same fields (`location`, `mood`, and the `haiku` output) typed as strings, but we now have the **optional** handy levers when field names don’t provide sufficient context for a task:

1. **Signature docstring** which DSPy uses as task instructions when preparing prompts.
2. **Field descriptions** to add nuance that might not fit within a field name.

Here’s our haiku writer string signature rephrased as a class-based signature:

In [4]:
class HaikuBot(dspy.Signature):
    """
    Write a classical haiku given the provided inputs.
    """
    location: str = dspy.InputField(desc="The setting of the poem")
    mood: str = dspy.InputField()
    haiku: str = dspy.OutputField()

![Class-based Signature](../assets/class-based_signature.png)

:::{.warning}

However, **resist the urge to restate what the signature already says** or write prescriptive tutorials. Expansive rules, watch-outs, and guidance are what optimizers are for (more on that later). 

Though it’s worth noting: field descriptions are **not touched by the optimizers**, so mind your naming. A poorly chosen field name can’t be adjusted by optimizers.

:::

We pass class-based signatures to modules just like we passed string signatures:

In [5]:
haiku_bot = dspy.Predict(HaikuBot)
result = haiku_bot(location="a quiet library", mood="mysterious")
print(result.haiku)

Silent pages whisper  
Shadows hide secrets within  
Quiet, yet profound


This call renders and sends similar instructions to the LM, with two exceptions. The docstring and any field descriptions are used when building the system instructions. Let's `inspect_history` to see it:

In [6]:
dspy.inspect_history(n=1)





[2026-06-16T07:37:31.320249]

System message:

Your input fields are:
1. `location` (str): The setting of the poem
2. `mood` (str):
Your output fields are:
1. `haiku` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## location ## ]]
{location}

[[ ## mood ## ]]
{mood}

[[ ## haiku ## ]]
{haiku}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Write a classical haiku given the provided inputs.


User message:

[[ ## location ## ]]
a quiet library

[[ ## mood ## ]]
mysterious

Respond with the corresponding output fields, starting with the field `[[ ## haiku ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## haiku ## ]]
Silent pages whisper  
Shadows hide secrets within  
Quiet, yet profound

[[ ## completed ## ]]







## Tightening signature fields with richer types


Sometimes a plain `str` is too loose. When a value should come from a small fixed set, we’d rather pin it down so the LM (and the caller) can’t drift outside it. This is the unit-test framing from Section 3 made stricter: not just *some string*, but *one of these specific strings*.

We can use `typing`, from Python’s standard lib, to add richer types.

Typing `season` as `Literal["spring", "summer", "autumn", "winter"]` does exactly that. DSPy now accepts only those four values, both at call time and when parsing the LM’s response.


In [ ]:
from typing import Literal

Season = Literal[
    "spring", "summer", "autumn", "winter",
]

class HaikuBot(dspy.Signature):
    """
    Write a classical haiku given the provided inputs.
    """
    location: str = dspy.InputField(desc="The setting of the poem")
    mood: str = dspy.InputField()
    season: Season = dspy.InputField()
    haiku: str = dspy.OutputField()

In [9]:
haiku_bot = dspy.Predict(HaikuBot)
result = haiku_bot(location="Bodega Bay", mood="mysterious", season="autumn")
print(result.haiku)

Shadows in the breeze  
Autumn’s secrets softly drift  
Bodega's hush calls


But if we pass `season="fall"`, we get a warning explaining the mismatch:

In [10]:
result = haiku_bot(location="Bodega Bay", mood="mysterious", season="fall")
print(result.haiku)

2026/06/16 07:39:13 WARNING dspy.predict.predict: Type mismatch for field 'season': expected Literal['spring', 'summer', 'autumn', 'winter'] based on given Signature, but the provided value is incompatible: fall.


Fog cloaks silent waves,  
secrets whisper in the breeze,  
leaves fall, shadows form.


See [Signatures in depth](https://dspy.ai/diving-deeper/signatures-in-depth/) for the rest of the surface — output validators, multi-output composition, and richer Pydantic patterns.
